# 09. Selección de modelo

**Fases del guía metodológica cubiertas: 15 (Model selection y decisión técnica)**

> Regla central: *definir -> auditar -> dividir -> aprender solo con train ->
> seleccionar con validación/CV -> comprobar una vez con test -> empaquetar -> monitorizar*.



## 15.1 Criterios

La elección final se basa en **validation/CV, no en test**. Tras el tuning (fase 14)
comparamos los candidatos tuneados y sin tunear con Repeated CV (3×5 folds):

| Criterio | Método |
|---|---|
| Métrica primaria | ROC-AUC (media y desviación entre folds) |
| Métricas secundarias | PR-AUC, F1, Brier, ECE sobre validation |
| Robustez | Repeated CV (3×5 folds) |
| Calibración | Curva de calibración + ECE |
| Interpretabilidad | SHAP (fase 18) |
| Coste/latencia | Tiempo de entrenamiento e inferencia |
| Mantenibilidad | Modelo con buen soporte y determinismo |

### 15.1.1 Preparación

Cargamos datos y los mejores parámetros de la fase 14 para LightGBM y RandomForest.


In [1]:

import sys, pathlib
ROOT = pathlib.Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import json, time
from src.data.load_data import load_processed
from src.features.build_features import add_domain_features
from src.models.train_model import get_model_factories, make_pipeline, _lightgbm, _xgboost
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, f1_score, brier_score_loss, average_precision_score
import numpy as np, pandas as pd

d = load_processed()
Xtr = add_domain_features(d["X_train"]); ytr = d["y_train"]
Xva = add_domain_features(d["X_val"]);   yva = d["y_val"]
print("OK", Xtr.shape, Xva.shape)


OK (396, 37) (133, 37)



### 15.1.2 Comparación final de candidatos

Construimos los 5 candidatos finales: Logistic Regression (baseline de ML), RandomForest
tuneado, LightGBM tuneado, XGBoost y CatBoost. Para cada uno ejecutamos **Repeated CV
(3×5)** y además entrenamos sobre train para evaluar sobre **validation** (ROC-AUC, PR-AUC,
F1, Brier). La tabla resultante combina la estimación de generalización (CV) con el
comportamiento concreto en validation, que es el conjunto que usaremos para elegir
umbral y modelo.


In [2]:

# Modelos candidatos finales: baseline de ML + top-3 de la fase 06, con los
# mejores parámetros de la fase 14 para LightGBM y RandomForest
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb

best_lgbm = json.loads((ROOT / "configs" / "best_params_lgbm.json").read_text(encoding="utf-8"))
best_rf = json.loads((ROOT / "configs" / "best_params_rf.json").read_text(encoding="utf-8"))

candidates = {
    "LogisticRegression": get_model_factories()["LogisticRegression"](),
    "RandomForest_tuned": RandomForestClassifier(**best_rf),
    "LightGBM_tuned": lgb.LGBMClassifier(**best_lgbm),
    "XGBoost": _xgboost(42),
    "CatBoost": get_model_factories()["CatBoost"](),
}

rcv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
rows = []
for name, model in candidates.items():
    t0 = time.time()
    pipe = make_pipeline(model)
    scores = cross_val_score(pipe, Xtr, ytr, cv=rcv, scoring="roc_auc", n_jobs=1)
    pipe.fit(Xtr, ytr)
    p = pipe.predict_proba(Xva)[:, 1]
    rows.append({
        "modelo": name,
        "roc_auc_cv": round(scores.mean(), 4),
        "roc_auc_cv_std": round(scores.std(), 4),
        "roc_auc_val": round(roc_auc_score(yva, p), 4),
        "pr_auc_val": round(average_precision_score(yva, p), 4),
        "f1_val": round(f1_score(yva, (p >= 0.5).astype(int)), 4),
        "brier_val": round(brier_score_loss(yva, p), 4),
        "tiempo_s": round(time.time() - t0, 1),
    })
sel_df = pd.DataFrame(rows).sort_values("roc_auc_cv", ascending=False)
sel_df


[LightGBM] [Info] Number of positive: 124, number of negative: 192
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000226 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 146
[LightGBM] [Info] Number of data points in the train set: 316, number of used features: 47
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[LightGBM] [Info] Number of positive: 125, number of negative: 192
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000107 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 146
[LightGBM] [Info] Number of data points in the train set: 317, number of used features: 47
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[

[LightGBM] [Info] Number of positive: 124, number of negative: 192
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000211 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 145
[LightGBM] [Info] Number of data points in the train set: 316, number of used features: 47
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf

[LightGBM] [Info] Number of positive: 125, number of negative: 192
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000164 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 147
[LightGBM] [Info] Number of data points in the train set: 317, number of used features: 48
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[

[LightGBM] [Info] Number of positive: 125, number of negative: 192
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000085 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 144
[LightGBM] [Info] Number of data points in the train set: 317, number of used features: 47
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[LightGBM] [Info] Number of positive: 156, number of negative: 240
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001330 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 152
[LightGBM] [Info] Number of data points in the train set: 396, number of used features: 50
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf

,modelo,roc_auc_cv,roc_auc_cv_std,roc_auc_val,pr_auc_val,f1_val,brier_val,tiempo_s
1,RandomForest_tuned,0.8382,0.0489,0.8452,0.8251,0.7234,0.1467,15.9
2,LightGBM_tuned,0.8369,0.0521,0.8124,0.7902,0.6923,0.1690,4.5
4,CatBoost,0.8309,0.0505,0.8222,0.8054,0.6804,0.1624,6.8
0,LogisticRegression,0.8287,0.0428,0.8302,0.7800,0.6981,0.1647,1.4
3,XGBoost,0.8250,0.0518,0.8129,0.7960,0.6869,0.1726,3.9



### 15.1.3 Visualización de la selección

Representamos la media de ROC-AUC de Repeated CV con su desviación (barras de error).
La línea discontinua marca el criterio mínimo de aceptación (0.75). La figura permite
ver de un vistazo qué modelos superan holgadamente el umbral y cuáles son estables.


In [3]:

# Decisión final documentada
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9, 4))
sel_df.set_index("modelo")["roc_auc_cv"].plot.bar(ax=ax, yerr=sel_df.set_index("modelo")["roc_auc_cv_std"], capsize=4)
ax.set_ylabel("ROC-AUC (Repeated CV 3x5)"); ax.set_ylim(0.5, 1.0); ax.axhline(0.75, ls="--", color="gray")
plt.tight_layout(); plt.savefig(ROOT / "reports" / "figures" / "09_model_selection.png", dpi=120)
plt.show()


C:\Users\sgml1\AppData\Local\Temp\ipykernel_8976\1705156372.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



### 15.2 Decisión técnica

Elegimos **RandomForest (tuneado)** como modelo final:

- **Mejor ROC-AUC en Repeated CV** (ver tabla) y buen rendimiento en validation.
- Baja desviación entre folds (estabilidad), sin hiperparámetros sensibles al dataset.
- `class_weight="balanced"` (elegido por Optuna) mitiga el desbalance.
- Explicable con SHAP (fase 18) y sin dependencias externas (scikit-learn puro -> fácil de mantener).

**Alternativas descartadas**: LightGBM tuneado quedó muy cerca en CV; CatBoost y XGBoost
competitivos pero con más dependencias. La decisión se congela aquí; **no** se vuelve a
mirar el test.
